# EU AI Act Consultation Analysis - Example Walkthrough

This notebook demonstrates the key analysis techniques used in this project, including:
- Data loading and preprocessing
- Geographic distribution analysis
- Stakeholder comparison (NGO vs Trade Union)
- Interactive visualizations

## Project Overview

This analysis examines responses from the EU's 2020 public consultation on AI regulation, comparing perspectives between NGOs and Trade Unions across multiple policy dimensions.

## 1. Setup and Dependencies

First, let's import the required libraries:

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Geographic data
import pycountry

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ All libraries imported successfully")

## 2. Data Collection

This project uses data collected via the [haveyoursay-analysis](https://github.com/spunfromsun/haveyoursay-analysis) toolkit.

### Installing the Data Collection Toolkit

```bash
# Install the toolkit
pip install git+https://github.com/spunfromsun/haveyoursay-analysis.git

# Fetch consultation data for the EU AI Act (Publication ID: 14488)
haveyoursay-analysis fetch --publication-id 14488 --out data/phase1

# Download attachments
haveyoursay-analysis download --attachments-csv data/phase1/attachments.csv --out data/phase1/files
```

The toolkit provides:
- ✅ Automated API data fetching
- ✅ CSV exports for analysis
- ✅ Attachment downloads with retry logic
- ✅ Docker support for reproducibility

## 3. Example: Geographic Distribution Analysis

Let's create a simple example showing how to analyze the geographic distribution of survey responses.

### Sample Data Structure

For demonstration purposes, here's what the consultation data looks like:

In [ ]:
# Example data structure (simulated for demonstration)
sample_data = {
    'Country': ['Belgium', 'Germany', 'France', 'Spain', 'Netherlands', 'Italy', 'United States'],
    'Organization_Type': ['NGO', 'Trade Union', 'NGO', 'NGO', 'Trade Union', 'NGO', 'NGO'],
    'Response_Count': [15, 8, 12, 10, 6, 9, 4],
    'Legislation_Sentiment': ['Positive', 'Positive', 'Neutral', 'Positive', 'Positive', 'Neutral', 'Positive']
}

df_sample = pd.DataFrame(sample_data)
print("Sample consultation data:")
df_sample

### Converting Country Names to ISO Codes

For geographic visualizations, we need ISO Alpha-3 country codes:

In [ ]:
def get_iso_alpha3(country_name):
    """Convert country name to ISO Alpha-3 code."""
    try:
        country = pycountry.countries.search_fuzzy(country_name)[0]
        return country.alpha_3
    except:
        return None

# Apply conversion
df_sample['ISO_Code'] = df_sample['Country'].apply(get_iso_alpha3)
print("Country codes added:")
df_sample[['Country', 'ISO_Code']]

### Creating a Geographic Choropleth Map

Now let's visualize the geographic distribution:

In [ ]:
# Aggregate responses by country
country_counts = df_sample.groupby(['Country', 'ISO_Code'])['Response_Count'].sum().reset_index()

# Create choropleth map
fig = px.choropleth(
    country_counts,
    locations='ISO_Code',
    color='Response_Count',
    hover_name='Country',
    hover_data={'ISO_Code': False, 'Response_Count': True},
    title='EU AI Act Consultation - Response Distribution by Country',
    color_continuous_scale='Blues',
    labels={'Response_Count': 'Number of Responses'}
)

fig.update_geos(
    showcountries=True,
    countrycolor="lightgray",
    projection_type="natural earth",
    scope='europe'
)

fig.update_layout(
    height=600,
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig.show()

print("\n📊 This type of visualization appears in the full analysis as:")
print("   - visualizations/world_map_survey_responses.png")
print("   - visualizations/world_map_by_org_type.png")

## 4. Example: Stakeholder Comparison (Radar Chart)

A key feature of this analysis is comparing priorities between NGOs and Trade Unions:

In [ ]:
# Example Likert scale data (1-5 scale)
policy_dimensions = ['Transparency', 'Risk Assessment', 'Biometric ID', 'Fundamental Rights', 'Enforcement']

ngo_scores = [4.5, 4.8, 4.9, 5.0, 4.3]
trade_union_scores = [4.2, 4.5, 4.1, 4.8, 3.9]

# Create radar chart
fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=ngo_scores,
    theta=policy_dimensions,
    fill='toself',
    name='NGOs',
    line_color='#1f77b4'
))

fig.add_trace(go.Scatterpolar(
    r=trade_union_scores,
    theta=policy_dimensions,
    fill='toself',
    name='Trade Unions',
    line_color='#ff7f0e'
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 5]
        )
    ),
    title='Policy Priorities: NGO vs Trade Union Comparison',
    height=500
)

fig.show()

print("\n📊 Similar radar charts in the full analysis:")
print("   - S2EOT2_radar_chart_ngo_vs_trade_union.png")
print("   - S2EOT5_radar_chart_ngo_vs_trade_union_image2_regulatory frameworks.png")
print("   - S1EOE1_radar_chart_ngo_vs_trade_union_image3_Importance of AI actions.png")

## 5. Example: Sentiment Analysis (Stacked Bar Chart)

The analysis includes categorical sentiment analysis across policy domains:

In [ ]:
# Example sentiment data
sentiment_data = pd.DataFrame({
    'Organization_Type': ['NGO', 'NGO', 'NGO', 'Trade Union', 'Trade Union', 'Trade Union'],
    'Sentiment': ['Positive', 'Neutral', 'Negative', 'Positive', 'Neutral', 'Negative'],
    'Count': [45, 8, 3, 22, 5, 1]
})

# Calculate percentages
sentiment_pivot = sentiment_data.pivot_table(
    index='Organization_Type', 
    columns='Sentiment', 
    values='Count', 
    fill_value=0
)

sentiment_pct = sentiment_pivot.div(sentiment_pivot.sum(axis=1), axis=0) * 100

# Create stacked bar chart
fig = go.Figure()

colors = {'Positive': '#2ecc71', 'Neutral': '#95a5a6', 'Negative': '#e74c3c'}

for sentiment in ['Positive', 'Neutral', 'Negative']:
    if sentiment in sentiment_pct.columns:
        fig.add_trace(go.Bar(
            name=sentiment,
            x=sentiment_pct.index,
            y=sentiment_pct[sentiment],
            marker_color=colors[sentiment]
        ))

fig.update_layout(
    barmode='stack',
    title='Sentiment on AI Legislation (% Distribution)',
    xaxis_title='Organization Type',
    yaxis_title='Percentage',
    height=400,
    yaxis_range=[0, 100]
)

fig.show()

print("\n📊 Stacked bar charts in the full analysis cover:")
print("   - S2EOT2: Legislation sentiment")
print("   - S2EOT3: New rules for AI systems")
print("   - S2EOT4: Approach to high-risk AI applications")
print("   - S2EOT6: Biometric identification")
print("   - S2EOT8: EU values and fundamental rights")
print("   - S3SLI1: Legal AI applications")
print("   - S3SLI2: New risk assessment frameworks")
print("   - S3SLI3: Updating EU legislation")

## 6. Machine Learning: Topic Modeling with BERTopic

The full analysis uses BERTopic for topic modeling of open-ended responses. Here's a simplified example:

In [ ]:
# Example open-ended responses
sample_responses = [
    "AI systems should be transparent and explainable to users",
    "We need strict regulations for biometric identification systems",
    "High-risk AI applications require robust oversight mechanisms",
    "Fundamental rights must be protected in AI deployment",
    "Risk assessment frameworks should be standardized across EU",
    "Transparency requirements are essential for AI accountability",
    "Biometric surveillance poses serious privacy concerns",
    "We support comprehensive risk assessment for AI systems"
]

print("Sample open-ended responses from consultation:")
for i, response in enumerate(sample_responses, 1):
    print(f"{i}. {response}")

print("\n📊 In the full analysis, BERTopic is applied to:")
print("   - 9 open-ended survey response fields")
print("   - Multilingual sentence transformers (paraphrase-multilingual-MiniLM-L12-v2)")
print("   - Custom stop words for European policy text")
print("   - Bigram analysis for phrase extraction")

## 7. Running the Full Analysis

To run the complete analysis pipeline:

```bash
python Phase1_Survey_Analysis.py
```

This script (2,500+ lines) will:
1. Load consultation data from Excel files
2. Process 32+ Likert scale questions
3. Generate 13 high-resolution visualizations
4. Run BERTopic topic modeling
5. Perform statistical analysis and priority gap calculations
6. Export all results to the `visualizations/` folder

### Expected Output Files:

```
visualizations/
├── world_map_survey_responses.png
├── world_map_by_org_type.png
├── S2EOT2_radar_chart_ngo_vs_trade_union.png
├── S2EOT2_stacked_bar_legislation_sentiment.png
├── S2EOT3_stacked_bar_NewRulesAISystems_sentiment.png
├── S2EOT4_stacked_bar_ApproachToHighRiskAIApps_sentiment.png
├── S2EOT5_radar_chart_ngo_vs_trade_union_image2_regulatory frameworks.png
├── S2EOT6_stacked_bar_Biometrics_sentiment.png
├── S2EOT8_stacked_bar_EUValues_sentiment.png
├── S3SLI1_stacked_bar_LegalAI_sentiment.png
├── S3SLI2_stacked_bar_NewRiskAssess_sentiment.png
├── S3SLI3_stacked_bar_UpdatingEULegislation_sentiment.png
└── S1EOE1_radar_chart_ngo_vs_trade_union_image3_Importance of AI actions.png
```

## 8. Key Findings Summary

The analysis reveals systematic differences between NGO and Trade Union perspectives:

### NGO Priorities:
- ⚡ Stronger emphasis on **biometric identification** restrictions
- ⚡ Higher priority for **fundamental rights** protection
- ⚡ More conservative stance on high-risk AI applications

### Trade Union Priorities:
- ⚡ Focus on **workplace AI** implications
- ⚡ Emphasis on **risk assessment** standardization
- ⚡ Interest in **enforcement mechanisms**

### Common Ground:
- ✅ Both groups support transparency requirements
- ✅ Agreement on need for EU-wide regulation
- ✅ Shared concern about high-risk AI systems

---

## Next Steps

To explore the full analysis:

1. **View the visualizations**: Check the `visualizations/` folder for all charts
2. **Read the code**: Examine `Phase1_Survey_Analysis.py` for detailed implementation
3. **Run your own analysis**: Use the [haveyoursay-analysis toolkit](https://github.com/spunfromsun/haveyoursay-analysis) to fetch data for other consultations
4. **Contribute**: Open issues or pull requests to improve the analysis

---

**Built with:** Python, Pandas, Plotly, BERTopic, Sentence Transformers, scikit-learn